# 🗺️ Address Data Cleaning — Density Map Project

**Project:** User Density Map by Clinic  
**Author:** Fredys Caballero — BI Analyst  
**Context:** Movet Veterinary Clinic Network  
**Purpose:** Clean and standardize patient address data (`partner_street`) extracted from Odoo ERP for geolocation and density mapping.

---

## 🎯 Problem

Addresses entered by clinic staff are inconsistent:
- Mixed abbreviations: `CLL`, `CL`, `CALLE`, `Calle` all mean the same thing
- Extra spaces, special characters, typos
- Placeholder values entered as fake addresses (`1#1-1`, `11#11-11`)
- Email addresses entered in the address field
- Null values and entries too short to geocode

## ✅ Solution

A standardization pipeline that:
1. Removes nulls and invalid entries
2. Normalizes street type abbreviations to full words
3. Removes special characters and cleans whitespace
4. Filters out fake addresses and email entries
5. Validates minimum length for geocodability
6. Exports clean dataset for map visualization

---

## 📋 How to use this template

1. Change `CLINIC_NAME` and `FILE_PATH` in **Cell 2 — Configuration**
2. Run all cells in order (`Runtime → Run all`)
3. The cleaned file downloads automatically at the end

**Input:** CSV from Odoo ERP with patient records  
**Output:** Clean CSV ready for geocoding + density map

---
## 📦 Cell 1 — Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings

from wordcloud import WordCloud
from google.colab import files

warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

---
## ⚙️ Cell 2 — Configuration

**Only change this cell when running for a different clinic.**

In [ ]:
# ── CONFIGURATION — edit only this cell ─────────────────────────────────────

# Clinic name (used in output filename and report headers)
CLINIC_NAME = 'Calle 80'

# Path to the raw CSV exported from Odoo
# Google Drive path: /content/drive/MyDrive/BI - Fredys Caballero/MOVET/...
FILE_PATH = '/content/drive/MyDrive/BI - Fredys Caballero/MOVET/Insfraestructura/Datasets/DSDIC25_Cll80.csv'

# Column that contains the street address
ADDRESS_COLUMN = 'partner_street'

# Minimum character length to consider an address geocodable
# Addresses shorter than this are likely incomplete (e.g. 'CR 5', 'CL 10')
MIN_ADDRESS_LENGTH = 12

# ── Address standardization rules (Colombian street format) ──────────────────
# Key: regex pattern to find | Value: standardized replacement
# These cover the most common abbreviations used in Bogotá clinic data
ADDRESS_REPLACEMENTS = {
    r'\s+':             ' ',           # Multiple spaces → single space
    r',+':              ',',           # Multiple commas → single comma
    r'#+':              '#',           # Multiple # → single #
    r'\s,':             '',            # Space before comma → remove
    r',':               '',            # Remove remaining commas
    r'\bCLL\.?\b':      'CALLE',       # CLL / CLL. → CALLE
    r'\bCL\.?\b':       'CALLE',       # CL / CL.   → CALLE
    r'\bCRR\.?\b':      'CARRERA',     # CRR / CRR. → CARRERA
    r'\bCRA\.?\b':      'CARRERA',     # CRA / CRA. → CARRERA
    r'\bCR\b':          'CARRERA',     # CR         → CARRERA
    r'\bKR\.?\b':       'CARRERA',     # KR / KR.   → CARRERA
    r'\bAV\.?\b':       'AVENIDA',     # AV / AV.   → AVENIDA
    r'\bDG\.?\b':       'DIAGONAL',    # DG / DG.   → DIAGONAL
    r'\bTR\.?\b':       'TRANSVERSAL', # TR / TR.   → TRANSVERSAL
}

# Fake/placeholder address patterns to remove
# These are entered by staff when the real address is unknown
FAKE_ADDRESS_PATTERNS = r'1#1-1|1#1|11#11-11|3#3-3|1#2-1'

# Keywords and symbols that indicate an email was entered instead of an address
EMAIL_KEYWORDS     = ['gmail', 'hotmail', 'outlook', 'yahoo', 'email']
EMAIL_PATTERN      = r'@|' + '|'.join(EMAIL_KEYWORDS)

print(f'✅ Configuration loaded for clinic: {CLINIC_NAME}')
print(f'   File: {FILE_PATH.split("/")[-1]}')

---
## 📥 Cell 3 — Load Raw Data

In [ ]:
# Load CSV — partner_vat as string to preserve leading zeros (tax IDs)
df_raw = pd.read_csv(FILE_PATH, sep=',', dtype={'partner_vat': str})

print(f'✅ File loaded: {FILE_PATH.split("/")[-1]}')
print(f'   Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'   Columns: {list(df_raw.columns)}')
print(f'\n   Address column ({ADDRESS_COLUMN}) — null count: '
      f'{df_raw[ADDRESS_COLUMN].isnull().sum():,}')

df_raw.head(10)

---
## 🧹 Cell 4 — Cleaning Pipeline

All cleaning steps in a single transparent pipeline.

In [ ]:
# ── Step counters for the report ─────────────────────────────────────────────
counts = {'raw': len(df_raw)}

# ── STEP 1: Remove rows with null address ─────────────────────────────────────
df = df_raw.dropna(subset=[ADDRESS_COLUMN]).reset_index(drop=True)
counts['after_null_drop'] = len(df)

# ── STEP 2: Standardize street abbreviations ──────────────────────────────────
# Convert to uppercase first so regex matches are case-insensitive
df[ADDRESS_COLUMN] = df[ADDRESS_COLUMN].str.upper()
df[ADDRESS_COLUMN] = df[ADDRESS_COLUMN].replace(ADDRESS_REPLACEMENTS, regex=True)

# ── STEP 3: Remove unwanted special characters ────────────────────────────────
# Keep: letters, digits, spaces, # and - (used in Colombian address format)
# Remove: everything else (accented chars, punctuation, etc.)
df[ADDRESS_COLUMN] = df[ADDRESS_COLUMN].replace(r'[^\w\s#-]', '', regex=True)

# ── STEP 4: Normalize spacing around address separators ───────────────────────
# Colombian format: CALLE 80#12-35 (no spaces around # and -)
df[ADDRESS_COLUMN] = df[ADDRESS_COLUMN].replace(r'\s*#\s*', '#', regex=True)
df[ADDRESS_COLUMN] = df[ADDRESS_COLUMN].replace(r'\s*-\s*', '-', regex=True)

# ── STEP 5: Strip leading/trailing whitespace ─────────────────────────────────
df[ADDRESS_COLUMN] = df[ADDRESS_COLUMN].str.strip()

# ── STEP 6: Remove entries that are only digits ───────────────────────────────
# E.g. '12345' — a number alone is not a valid address
df = df[~df[ADDRESS_COLUMN].str.isdigit()]
counts['after_digits_drop'] = len(df)

# ── STEP 7: Remove fake placeholder addresses ─────────────────────────────────
# Patterns like '1#1-1' are used when staff don't have the real address
df = df[~df[ADDRESS_COLUMN].str.contains(FAKE_ADDRESS_PATTERNS, regex=True, na=False)]
counts['after_fake_drop'] = len(df)

# ── STEP 8: Remove email addresses entered in the address field ───────────────
df = df[~df[ADDRESS_COLUMN].str.contains(EMAIL_PATTERN, case=False, na=False)]
counts['after_email_drop'] = len(df)

# ── STEP 9: Enforce minimum address length ────────────────────────────────────
# Short strings like 'CR 5' are technically valid but can't be geocoded
df = df[df[ADDRESS_COLUMN].str.len() >= MIN_ADDRESS_LENGTH].reset_index(drop=True)
counts['after_length_filter'] = len(df)

print(f'✅ Cleaning pipeline complete for: {CLINIC_NAME}')
print()
print(f'   Step 1 — Remove nulls       : {counts["raw"]:>6,} → {counts["after_null_drop"]:>6,}'
      f'  ({counts["raw"] - counts["after_null_drop"]:>4,} removed)')
print(f'   Step 2-5 — Standardize text : (in-place transformations)')
print(f'   Step 6 — Remove digit-only  : {counts["after_null_drop"]:>6,} → {counts["after_digits_drop"]:>6,}'
      f'  ({counts["after_null_drop"] - counts["after_digits_drop"]:>4,} removed)')
print(f'   Step 7 — Remove fake addrs  : {counts["after_digits_drop"]:>6,} → {counts["after_fake_drop"]:>6,}'
      f'  ({counts["after_digits_drop"] - counts["after_fake_drop"]:>4,} removed)')
print(f'   Step 8 — Remove emails      : {counts["after_fake_drop"]:>6,} → {counts["after_email_drop"]:>6,}'
      f'  ({counts["after_fake_drop"] - counts["after_email_drop"]:>4,} removed)')
print(f'   Step 9 — Min length filter  : {counts["after_email_drop"]:>6,} → {counts["after_length_filter"]:>6,}'
      f'  ({counts["after_email_drop"] - counts["after_length_filter"]:>4,} removed)')
print()
total_removed = counts['raw'] - counts['after_length_filter']
pct_kept      = counts['after_length_filter'] / counts['raw'] * 100
print(f'   ─────────────────────────────────────────')
print(f'   TOTAL REMOVED : {total_removed:,} ({100 - pct_kept:.1f}%)')
print(f'   FINAL DATASET : {counts["after_length_filter"]:,} rows ({pct_kept:.1f}% of raw)')

---
## 🔍 Cell 5 — Data Quality Check

In [ ]:
print(f'QUALITY CHECK — {CLINIC_NAME}')
print('=' * 60)

# Top 20 most frequent addresses
print('\nTop 20 most frequent addresses:')
print(df[ADDRESS_COLUMN].value_counts().head(20).to_string())

# Remaining nulls
print(f'\nRemaining nulls in {ADDRESS_COLUMN}: {df[ADDRESS_COLUMN].isnull().sum()}')

# Address length stats
lengths = df[ADDRESS_COLUMN].str.len()
print(f'\nAddress length stats:')
print(f'  Min    : {lengths.min()}')
print(f'  Max    : {lengths.max()}')
print(f'  Mean   : {lengths.mean():.1f}')
print(f'  Median : {lengths.median():.0f}')

print(f'\nSample of cleaned addresses:')
df[[ADDRESS_COLUMN]].head(15)

---
## ☁️ Cell 6 — Word Cloud Visualization

Visual check to confirm the most common street types are now standardized.

In [ ]:
def plot_wordcloud(series, title, figsize=(12, 5)):
    """Generate a word cloud from a pandas Series of address strings."""
    text      = " ".join(series.astype(str))
    wordcloud = WordCloud(
        width            = 900,
        height           = 400,
        background_color = 'white',
        colormap         = 'Blues',
        max_words        = 80
    ).generate(text)

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()


plot_wordcloud(
    df[ADDRESS_COLUMN],
    title=f'Most Frequent Terms in Addresses — {CLINIC_NAME}'
)

print('💡 Tip: CALLE, CARRERA, AVENIDA, DIAGONAL, TRANSVERSAL should be')
print('        the dominant terms. If abbreviations are still visible,')
print('        add them to ADDRESS_REPLACEMENTS in Cell 2.')

---
## 📊 Cell 7 — Exploratory Analysis

In [ ]:
# ── Street type distribution ──────────────────────────────────────────────────
street_types = ['CALLE', 'CARRERA', 'AVENIDA', 'DIAGONAL', 'TRANSVERSAL']

dist = {}
for st in street_types:
    count     = df[ADDRESS_COLUMN].str.contains(st, na=False).sum()
    dist[st]  = count

other = len(df) - sum(dist.values())
dist['OTHER'] = other

dist_series = pd.Series(dist).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#2980b9' if k != 'OTHER' else '#bdc3c7' for k in dist_series.index]
bars   = axes[0].bar(dist_series.index, dist_series.values,
                      color=colors, edgecolor='white', width=0.6)
axes[0].set_title(f'Address Distribution by Street Type\n{CLINIC_NAME}',
                   fontweight='bold')
axes[0].set_ylabel('Number of Addresses')
axes[0].set_xlabel('Street Type')
for bar, val in zip(bars, dist_series.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 0.5,
                  f'{val:,}', ha='center', va='bottom',
                  fontsize=9, fontweight='bold')

# Address length histogram
df[ADDRESS_COLUMN].str.len().hist(
    bins=30, ax=axes[1], color='#27ae60',
    edgecolor='white'
)
axes[1].axvline(x=MIN_ADDRESS_LENGTH, color='red',
                 linestyle='--', linewidth=1.5,
                 label=f'Min length ({MIN_ADDRESS_LENGTH})')
axes[1].set_title(f'Address Length Distribution\n{CLINIC_NAME}',
                   fontweight='bold')
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle(f'Cleaning Results — {CLINIC_NAME}',
              fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nStreet type counts:')
for k, v in dist_series.items():
    pct = v / len(df) * 100
    print(f'  {k:<15}: {v:>5,}  ({pct:.1f}%)')

---
## 💾 Cell 8 — Export Clean Dataset

In [ ]:
# Generate output filename from clinic name
# Spaces → underscore, uppercase for consistency
clinic_slug   = CLINIC_NAME.replace(' ', '_').upper()
output_name   = f'clean_addresses_{clinic_slug}.csv'

# Save and download
df.to_csv(output_name, index=False)

print(f'✅ File saved: {output_name}')
print(f'   Rows exported : {len(df):,}')
print(f'   Columns       : {list(df.columns)}')
print()
print('📥 Starting download...')

files.download(output_name)

print('✅ Download complete!')
print()
print('─' * 60)
print(f'FINAL SUMMARY — {CLINIC_NAME}')
print('─' * 60)
print(f'  Raw records           : {counts["raw"]:,}')
print(f'  Records after cleaning: {counts["after_length_filter"]:,}')
print(f'  Records removed       : {counts["raw"] - counts["after_length_filter"]:,}'
      f'  ({(counts["raw"] - counts["after_length_filter"]) / counts["raw"] * 100:.1f}%)')
print(f'  Output file           : {output_name}')
print('─' * 60)
print()
print('📌 Next step: load the clean CSV into your geocoding pipeline.')

---

## 📝 Notes for Future Use

### Adding a new clinic
1. Open this notebook
2. Go to **Cell 2 — Configuration**
3. Update `CLINIC_NAME` and `FILE_PATH`
4. Run all cells (`Runtime → Run all`)

### Extending the standardization rules
Add new patterns to `ADDRESS_REPLACEMENTS` in Cell 2:
```python
ADDRESS_REPLACEMENTS = {
    ...
    r'\bTV\.?\b': 'TRANSVERSAL',   # Add new abbreviation
    r'\bAK\b':    'AUTOPISTA',      # Example: Autopista Norte/Sur
}
```

### Adding fake address patterns
Add new patterns to `FAKE_ADDRESS_PATTERNS` in Cell 2:
```python
FAKE_ADDRESS_PATTERNS = r'1#1-1|1#1|11#11-11|3#3-3|1#2-1|YOUR_PATTERN_HERE'
```

### Understanding the output
The clean CSV contains the same columns as the raw file. Only the `partner_street` column is modified — all other columns (patient ID, name, phone, etc.) are preserved as-is.

---

**Author:** Fredys Caballero · BI Analyst  
**Project:** Movet Density Map · Infrastructure Dataset  
**Last updated:** 2025